# Agent Mots Tus — tout-en-un
1. Règle `REPO` (ton dépôt GitHub) et la voix en haut de la cellule.
2. Appuie **une seule fois** sur ▶ et laisse tourner (≈5 min). Les vidéos se téléchargent toutes seules.

In [ ]:
# ======== RÉGLAGES : modifie ici, puis appuie sur ▶ UNE seule fois ========
REPO = ""          # ex: "https://github.com/TON_PSEUDO/mon-repo"  (vide = tu envoies les fichiers à la main)
VOIX = "fr-FR-HenriNeural"   # voix d'homme : fr-FR-HenriNeural | fr-FR-RemyMultilingualNeural | fr-CA-AntoineNeural | fr-CA-JeanNeural | fr-BE-GerardNeural
GRAVE = 3          # 0 = normal, 2 à 4 = voix profonde
VITESSE = 0.9      # 0.85 = très lent, 1.0 = normal
CHEVALIER = "center"   # position du chevalier : center | left | right
FORMAT = "16:9"        # "16:9" (comme tes vidéos) ou "9:16" (la même vidéo centrée sur fond noir vertical)
FONDS = "fichiers"   # "fichiers" = tes fonds du dépôt | "genere" = l'agent crée les fonds par IA | "mixte" = les deux
NB_IMAGES = 8        # images générées par vidéo (mode genere / mixte)
MOTEUR_IMAGES = "pollinations"   # pollinations (gratuit, rapide) | sdxl (GPU T4, plus lent, en secours)
ACCENT = "rose"        # mot en couleur dans les sous-titres : rose | rouge | aucun
CLONER = False     # True = clone la voix du fichier ma_voix.* (TA voix ou une personne d'accord)
EXPRESSIVITE = 0.6 # clonage : 0.4 calme, 0.8 très expressif
# Réglages par vidéo (facultatif) : lignes @ au début d'un texte dans textes.txt, par ex.
#   @chevalier: chevalier2.png   @musique: musique3.mp3 (ou aucune)   @voix: fr-CA-JeanNeural   @grave: 4   @vitesse: 0.88
#   @fonds: genere   @style: (style des images, en anglais)   @scenes: lac au coucher du soleil; château dans la brume (en anglais)
#   @position: right   @taille: 0.9 (taille du chevalier)   @accent: rouge   @un_en_chiffre: non
# Sans réglage, chevaliers et musiques tournent automatiquement d'une vidéo à l'autre.
TEXTES_PAR_DEFAUT = """Avant toi, la pluie n'était qu'un bruit.
Maintenant, elle me parle de toi.
---
Il y a des mots qu'on garde au fond de soi,
parce que les dire les rendrait trop vrais.
"""
# =========================================================================
import os, glob, shutil, gzip, base64
from google.colab import files
def sh(c): get_ipython().system(c)

print("1/5 Installation…")
sh("pip -q install soundfile faster-whisper rembg onnxruntime edge-tts pillow")
sh("apt-get -qq install -y fonts-playfair-display fonts-dejavu > /dev/null")
if CLONER:
    sh("pip -q install chatterbox-tts")
if FONDS != "fichiers" and MOTEUR_IMAGES == "sdxl":
    sh("pip -q install diffusers transformers accelerate safetensors")

WORK = "/content/travail"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK + "/fonds")

print("2/5 Récupération de tes fichiers…")
if REPO:
    shutil.rmtree("/content/depot", ignore_errors=True)
    sh(f"git clone --depth 1 {REPO} /content/depot")
    sh(f"cp -r /content/depot/. {WORK}/ && rm -rf {WORK}/.git")
    os.chdir(WORK)
else:
    os.chdir(WORK)
    print("Envoie l'image du chevalier :")
    for i, n in enumerate(files.upload()): shutil.move(n, f"chevalier{i + 1}" + os.path.splitext(n)[1])
    print("Envoie tes arrière-plans (plusieurs possibles) :")
    for n in files.upload(): shutil.move(n, "fonds/" + n)
    print("Musique (facultatif, Annuler pour passer) :")
    try:
        for n in files.upload(): shutil.move(n, "musique_" + n)
    except Exception: pass

B = """
H4sIAAORAWsC/7Vd65LbxpX+P08BwysPOCI55GikKMzSiWzJsSrypSQlToozywKJJokQBGgA5MyYYdU+yv7aivY19Cb7JPt9p7tx
4WUsx1olngHQ3ae7T58+fa49n35yvsrS81EYn6t47Szv8lkSPzpxXffk2VTFueN+k+SZ83aVuU7PydVtrpzW5846CW+dZDKR5zB4
/y5x1irNw7EfKafbedq57f72ouMs0/f/gwbv/8tZJlmu0vbJyYtbtViiUu/EMb05PjtqL++cVisbp+ES/UlHWTu/zfFxNHUmSRxk
53iex+F0ljvjmVr7UQiIy3jqXAEW/7VaKp6GsXJUMMUwWxjlWDmTtPXV69bXKk7Db9Uq9SOULFZZOHb488eVai+Wj05OKl32nFVs
5rr0U2cUJeOm/Myc7P07fHr/LpOSFfqKwil+jpM4V7Efc7wtQV+4WCZp7vjpFDUzVbxnd/E4TOzrws9n9jnJ7FPqx0GyKN6Kxtls
lYdR8bYaLdNkrLKiXXZXPOZA8ySMiqZ5uCieV3E4TgIV+Ll/MkmTBeaSz6Jw5Jjy7zmokx+aztdN56vv3zh9h6vZlIVtOo86QPWn
TvdJ77eY9mKh0FdmqCA7efnNH4cv/voWbTZu++/Lqdt0+FvpB6yW/L5Ro6W7PfnLy+dl5cXyUsoWyVr/nq9t3QXqvnrx7DkqdtoX
j9l9hrnFWF1/TaRHviZJL2ucvH328pVUfOrUKoIY/5nVap6cfMrl+nX/HC5JmPthqrKTQE2cdBV740XQ6AlVphhKuVBtFg6yPPXG
DRB16oydMHZQ+7rpjP1lvkrVMFnly1Xef5uuVFOoUB4bAi0E+HaqUC/mCjqfYJ49Q/3oyw8z5bxexVzsF2mapF5Rxn/ul1gukBZ2
5LvxLFm9f6d6jus8xH9YoySMvQMjawx6jzqda9a6ilk5bWd5oNJ00Oo+7nR610UXeoh6dLoWpgIkEykBNl4eJrFHUjOoQSmQIxhx
JxNgaKS44C1ZdsXhy2s2S26G4BBpqDJ+wOCwa/oWotuszZHTbCUTVhxn6/6y38EjO72uDW8SJX7uYQQYJliO12iYgUbJ1FtkUzPE
ZRrGOd+baLHKZmYpPgbhlDzUe/v2TUM6z5agpGGmyEtALh5XvwkOswjz/sWjjh2Un4JFAnOqLQ281PV+/+/9QfuT3//vf/73deMq
e/iPq/ihq6mnnGCJ84FeNC7zksssEEs6WqLK0rYrvt7MsJMcbCZv2XA+16Pq1XA/FuDLdjoJ48BzmxhBxwy/UasIMmbdf3eedHq7
q7cLxTkKhavnL5cKtZaDHps9dLrX9fnWpjQwdXrXe5PDiJa9o9BrpFMS9fAmDXM1vPHX3ni2iudZ08nSJivwG8lulSmzaIa1xqsF
Djo/c+Jl9XOWrOKA7JpF2USKLOPqo277J5UmmUdizFLnzEBuOkF+t1R9V6j50YWrx7kMFYinvsx6N8sgK+ssFe0s0Yuf+Wnq33nj
fcjgOtnMXyqv1a2gtg7BjFgXZ5O2oEd4ikEJRoxecFaOfZ6WKNQAGkQct9Vw7ONo53mwNTjO82w4nvk5pIdRcmt2RIFhnK5ZHcF5
ko5neto82cqm7UUeDQHNVvyyKPlmFZGFx9OVH2ErSuMFuGuEcegBtacq99zxyC24sC4PM+fbJFYlRgO1RiN3vAp8l9VkNG2+tsNs
iLMqjPxRpLyGoyKwane8XLlFY9vn0ZG1OaXhEnQITh+rwEN3kHD6+FUuiR7xgKO9BjABKoV69etkkZEsDnGdRjmn+Q3aBOE49yKf
I5mqYRiAMsid1a0/nSrNiPtcjLa6xfiyLFyH+R3OtMl0eKMosvU77cvadpPaqZoMRUqr7735zcD1V0GYYLLJYpkPycBlOvVWJW/y
13auWKuYIwLhNZ2zs/lN2avGgKVWtGlnEP/UT1iPNlYCP2V7Wt5xYHvrHmqbXEak92OFZCmB/jyxshZpUpPc8hHmQNmr2C7tmzCf
DbPVZBLeepSQHhkK5PxQeeJuyBI8GUO2VCpwWk63AQbR7eC0eBhsH2jyivyMTJXEWqz9kGsPSRMDfVRZ7jy92+HqEBrQ1o61TSGC
EiRRrGco3cuCNGVkff4A+wvz8UxThTzWObIRhEUiYhftzF9rZoFpNnbY9+747b9Rqvx58UXdjtUyd17Ir5CaRebsUJYBVIdCcamd
RUotvYtih7PmvaLVxCVKWmQpfiFQ9ZwNG27tOhnhZmEE4NbdIQknxE9Mmo++fLm47HQ6UuSP+avrFmR0bakswHBxFgtlZQqHI3Bj
yQv6x2uQeCHsLiG7ONMUCHYyP87IFbHsqZSDxLNMtamyCGUAOxfYN47XKqA6507XICZfLC2RlgQa+wvluRxQGx9/8dRlCpjqpCrK
TVw/I58DNQk2zjaTXvtysm36YC8+VUj9velT00n6my5GaaqULBVd/QjeGeL47E/63W6nl/d/7N30u71p/1GzWvaEsmxZ2HpE8Wmx
NFKjVryommgSRQnPLDwRC3ZJFv5cacZ0z9YvKw1T/+ZgxZMqiyRaqweMrHqd+/Dj/hiOgu/V4Ou6lD0q1P7Ba7cDYo+ET3Yk80P0
XPQq8pW8btzy4HZ7O0IAB4GdZ77zcTvQZ49o/9el0nMQub9WeCe7wJGMpchwHuRa6btJ0mBI3pB5cnBp2dugusZSRSyZ+DSHDCFS
g2un9jj4Qb9+UxzZxyQR085tOODj1Vaemy38KAJ6wVKhRYJjixiHU+Kpuy8jWDh1QUEIXk1x3A2LQxU7MaZxZqTpX+aIHWAlAiMO
lFjIsUUzrcPuCfgH/oVxmId+ZI77PnE36IGCoBav/YCkBXRV9GA589GZiDPeTZvPVqbHMPAIjYYP4IKNiqSDWcnbjbxJs2x/fNga
dZDXVcFFGtVPFaMZlOAOn0Q0R6QK0hYsTKkYkvJQSjFtEgAeYxVF5eFEZXTiOs4ndpEx7AAPSRxCinS8DddCQUN/2rneYuKGMiHg
pMltCC05nJhVzxMR/bRCKEqj/Z77pK6Ff+sVOrpeXYgSnTYOpW67o+tqQU5QTi0wb0CXuhB05kQnu9CIysTmsfBMg0LzxABF/IRC
1+4UcgjaYaWmAuMnYJtwmrazimAScPwy3DOpfu6US1dR1jyBJ309RJsz9PXbijSBr5Bm9/W5X8kToMBlLawmDifn2Zs3whP8LJPN
4BV8wCAag+vUrRFajsOczp1HTyC6bXvmwwN55/cnEOg6FwFK+BWKc+dx+2KydQ3fh4U0H078RRjdefJMkblPgamUCb5PIhpEybiq
4+1hX6QgbAgImeN9H/l3E1iznOcgNDxDD8XKYENFEexFjUJQCE2f7Oce6dGag/SJMoaF14/l9GhpOw6fH2z0wLd4KWBeNwyhwo7Q
GHT2VfbdPVbvdwnsFyQG8KQuz92dHHt/laQ+fz9Xf/f/snLeEBfufQLxNKnYrcYtTGI8kylN9iaDfmE3642SKHCv92wgANSOkhuV
QvMQjpVRnPLQqvys9yrm39i3lFjq8Re/FCmmZX3OoKRnX3754tu3b8QgC4MDz1j3N08vnn/1FaeWJis5dt2L5/aTvxqvYnwioW3F
3jpOVhHM7LINHO+LP762ksloFUbBECPwhE9qq5zgCNiCMAVBDEf0eAw9tK87b9JtMLyF6vi4CaM1jICQAsLJJFX6KNCTm2JYS4Ab
r1LDXSoKrjD6HY4NSSK1LIqM7JbYpV1IWtyKpWSV1pRVwobh1JH6LHQ+7zuPePpqYA/FLnYjgD53Lp5IyYom0mt8w6/eNcG62kTX
/F3PPXgqcsCD7jU4r218cQ14UJ0f7yy/nrTlebXhahtaWir65kOh9xbyJT6WUA9A1GaDJEoIzJCGCCB6lZr220AvlyHwmfLJqksp
zB28EbeO8zKeJNdXsX57C8Gk56wv253Ow6uYG/O1yv7acx4Exdvf9NsPqb98k99BQHUurmIYoSuQ/3L50JGyDIC/Eo4CYoQ+As8F
OElcPGny+j7FqZjefYk5raDEv1GwRQWVDxXQ361yWF6ULfnCH8+LZ2znpvMSLDGEpezPMKinrAp4YFFz9R1Pujd0hf3V/P5bDfKb
pT/GMQ30xVOKzF+APFUqs2g6pls0nPlBcoNKdDAtBN3fQIAN41f24XUNqv72l6bzgq4BwK8hyuDvG9iNmg+y5oOg+dnXnc5X8m/3
Uf/D4xf2sdVtduR/MCvIfxXI8rl50b5oXjYfN3/T4f/xZXehXqwxheoivfLvFJdAS2ov4qDpGBTo5dufbDHDCtwXk4kao/1bCDZ0
TzxwPO23qrIVTZf6J5G7YwQLm86Uu1PB+qMtR3ovVLaccGe0mnI3d+kMoTuqKI5vpVBaDUIxQ9cqcrPJZ5i8ySgMfG0EnNqNbuo+
xH5/XDL0mJsJYp0XU5c5VPlJY7c2ZAxFhOphPxSnWUUESqpmwJoYNmxqs9C0zm1QJsLXvE3x1Yd+627I/cUsbD5s9YfdM67Gtnva
I8dlhhRS9RtCCHFxkECowy/l0q8MLXGPSeqB0PGxGsHt8R/eKv4H6je83/cHzfbvetoDcvZvDatlyqwmkT/N+mj1srE7rYLh4blR
kbthHsmsuKZtZCLyokEDcvZc3fUjfzEKfGfec2zJYH5dNY0bL80Omucypjq5CdjenodEGC9PHcy5X4yJH0yPPGse7YsF0rOdl7u5
uhp/9vWD7LPtg0w/663+2VZ2i/QiY9oxuZE2fwZ2DWeTW1q+0YMfP766wsHtgck8CBpXVxM/8B53mo87eB5Fq7TbvtB9U8T9gd4M
nvJAK9+/Fon9siqyy5Yt5vMcemIyXYGWOsLJsqYwNcOg8P4gO3y4ssdCJJedgS6LD1QUQSk0LlhnqMzVWiLF5CXOS+3WGFKV8uSs
055R3UbG2jCfYB83vLjvrvJJy6rgRvriKur6v1z/cKbv38Xv32mFTSR6eG7C9/9MVQv7ESY+7+WzxsnzF189+/Ort8M3b//26gXP
ZBcHj6JeOHaSMMJqAuVyFEHvhh8dn3EazsE84xz2WTCQZJLDPpcuIKtGgYoZ7TDLqxy4tO2skwgEjRNwjMoBLKZ3IuOhBxyPdzDV
wn9Kk1kyVygIYLld3IGxgRs4fr5IsuVMpeowaMCc6f1AzTAE68AcA6gSSzSYYUiAHyj44CMFrhcnxvCDhxvsL5w36VzeQDgJ1qDE
C8SXF5R2B1pF8mGRZtBLRrMnHPJZDkZF7d0ZR2Bgjj+CAZBvfrTAEOZ4zi1mYP5Njb3L1bOapgrfsem5bSnZQ+KMBS9EA6ZPXT2f
Uaqe4Tf4IRpkFkatDzSENVQDoSYGElIwfpAX+Hp9EuFSTja/swAmoYoCOJjRKgomolBoIwjX1KzjOILgBh6sblhxCRRGFiqDcjB7
WIoDCzG/SeQ9Q/cwIMnwE/QeOHx1EswhFXlbCAY9sobpKYW/L8xgfR6rguz0SWCh38EHOoWsDkIkEJ8OZ3g5ArqTQJbxWEwbEVdG
iqvoNZQKcxy3A5mMhRqkvqb3FM4Rg0YgycFCpprYF+gXpBPbiachiopVwJTSOwb/ZBnsdrJClQUZqYzU6MtCUeiI6K0ziLQwLCam
fuTf3hlKyKk26A7HSbbAAGM1WkW+bQSiggPFJwlL/ZlCdzpsSWafJhgOdENndGcpEJ1AYtQIL5aMxOLDCYnZ6/JMA6zu5wo5YmES
DMth93oXzMSeaJBJfEFrnWEUZnesKP2yKpYc1LQKDHwBHAtvGUFAmMtc9Yra5S92iwpChnM5Orgr446bYaurEdvptdaj1SQdK4Re
+WYAFsgNbOCcAIgd6JaFKldH9hKsZkGVcsDe57Z1dkOeKTtdr6TwjfIEDuM5qEblpHIJWMPi+MuSTLKQorzQGFUhjYIAuyqAZc8A
FuRxSpT+yk1FSgCF6bg2Uqa09Z3JCiu3SNBUSDVNNGkhCKOkrHRFN7BdES6znmCV/wCxkSqYgr/KVwsePfAFZQW92j1r9ir3D6cy
AqueFQQ5Vj6rrfUGCPyb2Cz+ktjRa1+lXnIKHpU5yUzRdl/usNGI85JNniRc2Zxu8WaFNxVTBLEU45QWP65ClZtlAn4C7s6Vqrci
l8CZkCY4ni2HtbsEZ3ZuKKnYyMVixKCdojfi3ddxGNh7lkrXWAFQABvHGl8Gd7ndYYB2fXLypxd/++G718/FmCJyL7gT4qDKIMps
/P6f2GCYjxOdAuiK21CPAw67UCwt/9r0VY7TXdofpi7LkTQXknnW6CqiDI7WOzRoNpIQsh4AcMOCWucQatn2l9ELK//iVlkSqTBi
M7O2WMyUfloZrd4EMq/66VsckLRTyTT/xZOyoFFjKfsVXGCBdSaI/98jNkIrGamezWrJufigSDBUGZ3Myx6mCkIWvNowctQXeKJW
gnPTu1qM7NaeIAgS4SSYouCPMk0gsldVRKnDghqlh/RRwMWrMD9O9h/GVKFbgPsRykc5mOCvrQPLaI50KKXnKrDz0tRQMiQKyEZw
qYh09IEqX7D/sURVyk+Cso8hs+KgF+r6taIrbUUERH5K2eum1iuB0FehgcgAjst/JiaN0BAngBmWUs0BQUiojiFv4uAFQy72pw4V
Fxb1cSQzSLSaa/z/yL2gDuGMv1L+jeGG07ztY52MmlAgD6QFNn+1pGCDBcVFNNTW6axwu1lnh1HRx2VAZCUQvh3TLsloEM/99qvn
NB1ZJ0xD27HKqgx9mibpHUOlEYHtfhO71smRoWuVDdGDCb2I6ZgAzS2qvjixvfNjJRYzSeiHjbALPV2mbQV1MwxF2/7uNBloUNQA
7Ry2Oq05YSuTtEFhi8zbNzoJW2MvxP8aOrO4ZQl03xbEr9Yss66GcYjlbHBbulN29G30c1sFfV0Js2LiQxvhNhOczh5BNXZRxBaD
XkwTKMv3w5rLQGV8bsDkWvGCsSIcwAS14wMmSLOIUH2HSzkaxLySeToswXqsMjCwehzfKkXlUVsneux/TxFoZEcq4R/fI0PBVHm5
wI6RElQWE80sz5dZ7/w8ZEm7Oo62H57roZwzEL/aa/vHVcJwWikd9OAzrrsbH8LH/HtYTuAT3vyw/WymYzI3X28/42z6G/7cfhYn
tJD0czjXPpO4jz6i3m+N1erXxBACA2haR0f7tf7t4XNTvEbY9/2N+2d4o1uSfET+8E3yE7nK+eN2x93WLZTci/tQ8ZowRArvEhCh
sLZ9ZCrQVA1DZB3EnlVvdAejtEfI9YrC6mWt2gJd2jCqI1zsbwxa+xdtwTao73H3onfQIHkgptCVRQf+kiVlMYxox6JeiZH6SBGP
lzai9ECAY5X4KpGN5S7JgtvoZ3dHGZK9DJdqJ5SJEMqwaqmwF1UteyaAVIMg27QI4H4jJ8Fz+YwB/vXV92hMe2olOl26O1pxL566
ni0DIXLE1J47bLtMYLQCC6R1G7VGiFBsITpG3AyY4lDHzOuYb5Eiuk/q0U9rP4XgAPf2ZNl9gmaYzzDzJwoHXwa5cifAicNHABM7
HspuHCJIeYiMEYCuErKN5hJUMpSLDbVrXLw2HM4fdVA0SYwx5402pA0EPw65Wp4smVmxqWlvVrVvFzdWU9DBWtkQLVcfbxX7KlmH
sR3AcgwKwYFKc+xqSj/cUiVM+5vAV5Ttp+xoztR9dHlJTiC86TdPnjaZLjEM4wkMw5DhhoiZQ1DZBepMVyElI3yie7X/BAECuyCn
dsb9aaMt2yqDT87OkukMPOvFW9ho6OBj2dQldYtzZljqalklmjIIUx1PeOh4NxWQwYafdCPwnDYJXeoWJ/wwmVfWWoctyqktYYzc
WCaUEXx7oITTKh3AUt2R9HHp7cOtI8lgu+11OJbIJJQbdoUTqT6aDgshxSSQAA/7PtJsXPdaaWgVrq8pQ4elZ+Nt09nouHRqoFu3
Elqo6KQ0Rz1/0e/TZXrh2VnFl8lAY4NJhva6mNYmlBAoSSOsyTcqNh5cweSO83L3MOI/b/eMFxQCCjxtdSQbJ61hdo2S2xlWd8BF
BuztpA4dj1n/QC5ejQZkPKA+JzbiVN5C8PadDUa/Fd6I7AtGB4Jh2/DA7gXiAysnyb5vj/mrYbzaDz106l2dbyhXmaXfOvBkIghx
zFjN62273W6UrJzynaDi3iB6HUCkTCfGqYU4M8dDUDFsONCBuHOdcdiCkytbZY26D016+DiJnHCqSZqxTgPkLq1u/Qk1udRQO0Od
ZYMsyxw6ESVMrXYKa8bIc892Tm854to6n8NqF2xrk2b/4diMWIrfgsE2Q1iq0WFu2zXxqAYFODlgSPBkTLUocES7Mm4EGWPM52rq
CEzsJgRMgMjMVDKEmMAv13esM7bbRqx/6YuVV7umgHR89OU6rye1+B8dzq859Sa7QQRjNtv2gDjw7yQNsWVxEPkIbx3nQ3Fo9sN4
jE2Sqaa7A2QMwYgCbA+Sa++2f+qFNy28Ns4vHm6KmW3PGC6BYP9mWdw48y7O8vNNsEX+2mlvF+4dQc1agIq6p00oyciE63ebE5w3
G6RAb5sm6/Rutb686CzdnaRXzf4obAzcVpQkSxt/gKBAKODCMF1JpuYnBj0GOizDxtEDs9cH9D2NyQJ5ZuIfhLoqqsoJ7XpXJ+7R
KTaOi/NrMvAiXBhj/9CoTGnYKY+CrDwIoGBzAB5yPRl0sWYqUyAxyI8ZLitnHD5+Lpjraq7c2Ue+tlIN7Rq09CJkPCx5LGU9CZr9
uVX4kASIs6W4q1trhn2uJ5L6EO9BrqSitMY9AQBt5fbiiSS7t5gspyQIFiaVO6YFyOdxKrGk3d/oRIki6QfVKU4MtUHS0xK3iH9D
nImWOx1VMUMmcu1qMczH5OUNnvv6j188Kxk4VJgpPV4+g9I9FyV8h9yQIoJAQlIZBHnxuCNx7QgtlZAe5gmIxDPmCcTYhPfvctgb
1XFqkvECKO55KG48YKZN3TaxkMAfycAJF41fpAEVUfRmLDxqjgTSP9GB9OY4gt2dZ1HOoHw4XvjL4AdmKca7HMYRC70Cj3wrByTz
QDNuT49FVvDWImi5llUWX3z9SFcWwHYKdNNWp28sUPSwDBGKpZKKjIsLGayMC5P13FAXHz9UstUXcPSlDUO7mIwmKWIndl2wLN3z
yyIX3mB3L5HKJBeW2VHrQCJcCx4kFeopDVKlCJPjrRAndcHGYXOur/S+Yf1euzvZZpVbTTYCTX92TUSsGfcFxl0JozdDt1kplYQg
M/jSTudnWQUpCGVD4rUJWtJhi5ShK/H8IkbzgxHPGS4lIdo0AIq3TZgkwmRoDuXbo0vyGTXRb0+emAQpzTYg9xqTWwwoe2HZeCzD
J4uuhyY+m+82/FcGYr7V4vxKIZDydOcDJMAKJkspcAfhj4BwkdDsHpxWlBSbzgZthrqy1pYmiBvCJTGiKS3CW1pTyrGw+cP+vqRn
wDSOgxXdUB0EaoYqCujPRGjxApmXzxCvWDlpzZjuUT4t1aCKa3cnWA5y41JPv7lGzdMaHSqJZlqTy9HNh6xJbcDg5v7fwRZ0/ojF
LB0Umb2eJ1khfp9GAiaWVqdvZgiBdBiQtgJlU7nxyRIZvpqAcwzFhF5eNOWGnPYYfl5m9nBntwRAA0jwBKD9YG+4YDD1QWMwC2gO
lg72LO/AyaB31BLNSrWMAN0NxXQW8TAEpctHRujupF2gRqEQ8nkJ3t+pRDhqYFW7tGSq1TXw0tQqE2jU7+swdAFRUU2tmswbdco8
wqpKoAcaXjftmkCb7eoo5QfI6eKm1RIWpOVaTmBFr61uzUtszQW4BE/NCYVSK0bESEI8eOeAqpKgqfZQJDgRxHT1OW4aIKcTnVMV
mbHV2h8kbEPyM3zcCn9F7qplifFUd+mbLnXXkDRPDg5QeLpusTAtCsO45RniEywnaev5Nch70A8LsMV4Bei1Rv5kXGLWmHndTs+c
riKJiUzM3SEX55jfEHwQObGW55n2GTruT/BahnFR7SDdwUZTrESF/HiHDWQaKIv1DVmR+EoxHwEFcqxzdJXg4bElrIk70Hbm68Em
3PbW17eE1Zf6kizZ3+TptmcP/v6GxdsexgA5ur/Bb1nkwS1aX1c4q8HPxGWBxtCnYFgUfCGcwvSDoAzejOTtXWnVcBjinaE8ZbZc
nPCmJxAwUzgxrHFOHyjad5qXj01K4kgnGFWIPodakwN/ErzhUcnptLuP9U/Im559e6RLOvqbfruUks7jKs7ZQ4kyOHlvIEr2kSrV
u8N/N/3wpjfrh7OzDXAlYWx9icJDhjvsBFFPW5X7pyOVI3AOweTNTd7Z4kd32zitnkf3dEPoXaw1evgo/R2iAbmcqmkctxyMhFYP
RlO7tHNruqCtorqhRWM2dXaEJdf7oXUDPd+tiEzuD2e4MKzTutGfjegkXy/01yMi1JzCmvt1a/YwnHgRUjabkMUa+PHo4uzrs2Vy
43Vb+TnlM1hVOg1X6A4nUnGDnfBNxcAw+BMcf0UlZbTKD6BEWBK3hLYGtC56m/msUNvT6cgfzOPrQ9gEvlhEtz0yq7B+p5v57fYU
i4iHOzzgji0YkTI4ygbJ2oKAYKqdzQybr2R6XF0RRefV5I8ev1xd9WznVPPLPNFCiKy4dGjV1cDFcFaKuLTMkzsh+VHrF7+o56L3
idsjuAz9YJLsbnvqWvE53x8e0lLXwD/Nv7/twTmjU1WMFqDvuuMdZFQXMugVlHiEFRT3HlrOwdK34fxtNTfCdtnUK8d79KBpPm0u
/UC/8H69XqfnJbNWOANp9mT3uPsridXBskBOBkfCvPC85exABcIl+xDQeljGjRWVIPwLP+wFyKJ8suFAdg09g2LBERUaDBeZ2VGi
OMlNMp1DG9TX1OhfMySPwRP9l/0WEPX2e/xqP+69ev0MF240MSyS3MbA3v6jeDpgi/KJkBuEnKkhGHy/cmgP9IFbuYBJTtEwE3l2
x11YG+fCjlPnJ/Q35TEKFTPChSJHUHdhEXcxQO3qWVJ24JpxSQ2oTLd9fZb3L8oTClFsuEmmCC/pdwa+BVY39u2DhY8uinT1DzdL
6RGINCTXJgx5FUSkblnpd4aVTsaNmmVq4YuwQVJoVl79a/e4CJUaGeuDLVyME1otavatp7pKeDucLKSONT7uQJUEbV9fHtIa6Vfs
mbkebbKWDC++PKQJTazlbmHEKK1njJQY8iLPuSc/y9T4V7zFM3P+MGaWJBgzIkN5pudyfynEAlanEPP+3TQShwSvbKAcgLtNxwxZ
MqyiSI9HJozfLLINN1t9gejcZrHrxG4pNpYio6hIA6oVSFeTzHKmuV1lZ3+4unmIX2R3UqeeES7xPX2TKaW1C5s9zxYVIZ7jGpTD
iPSD+wfwVGPSpwd5Xcu2N1YoPaXdXKeipsbxmqnkd9VrHmu2Yu1i1SmVlHOzf+UGSN20z3issVxw4h68ZOTYPZC1CzDASgLohJfM
vdZ3jrk2NMmOkCViG9st+ZnroczhEcY86ClaGnPTJtD2JLlT8la8dIS23XFtBYUvB7g2uPSXWnHWMT/P0umK6tz3fEvNhHxc4xfA
oGPKPLe4VddliDDCxxEJaEx2MxUt+9ZKogPTD190C6OhexT8iLwII/VxWx2giYXGwg7ojWSk4al2wZ/bfM9AyRl6H1SxuPAqG1EZ
sv6gZs/ZNcJcV8dQ1DsYeFObtr4Wxdgw4IHTcPFxjNnb+UenL5+hTHoSfZQTKCKjDoy+9MHXJnDQfV8dea3CfbihqibhJiEtcbb9
U4v3OFnAz1ygveJZ1RPSy3C8BxM2XgCuZRLaTmwib+bs93IPaBOFEEoHB0hQ7l9WNj0CmZngh+C6IYR/WQuy6uPgtYB+jNZloLj7
l/J2IYI3C2o32AlE+wNQLrTHwZQ4Y1InYnxzWaPG8XHg8KluC3ppw3uGfYBY5JKrplO9CqtKKlJ8FJ4IELV9uXsNdoF8MX/Lldne
jCLsPZMqbl4sGpeDw7LxFjymkSIFiyZPGN3NZcvdTgvXRmeNnxlvnSiKC3KB9r/jnMXQ1RTWWh0fgNg/fQZLF/dAlnsR7W6RsKwS
K7gb6OJoQ7m0sIrC1uXXPx3vh7ewHenmomCHuFKvlTPcZE3OEylzK5+nbnG5BYWJy3tmIrdMHp3J4+O0VbmT82jzJ0ebi6xcrIqJ
hPfk0tb7ts2HbZNCED86sO7l0cbU8Nx7S8VtcZBVdrtPj5OMiNcHV7J9WaylPs2D01Ws1UHvPvo2xrfDU9yDCfyVFq/sfshac61x
Duqr5ByiyVZ5hhQchaT9OjVI5h4ec/VPceFPFaZUOQqz5hc6vBBm8qddnKzgIIjD5SUQ53IPBIgIpt3Tn+H22tpzFL1PL3+uPfTi
2rSNragwEFmTUHXeps49RFSyndqExR+jW6Vi5UdjrZrw3TO6njVMSLjcQT3XuigYwllUrbutzN9DoFWjcooU0UX123Trnw/dCynW
/m8RaN6rXKysrxStnA36RvyMd42a60fJG96/w+zztlVlRRsSI+nIahBiKB2Jubm82Zy6D6RP/Pw3ILO0FRmZVkd168sYKreg7Ny6
ULkC5BsJJBnVb/azbjEZ065U/+YOcaeLF7dUpp6JX0xLKQjSXq3fv2u7jSP3aRZXdWi4vK2jex/w6iGIeFsI/szbRtzBKY6IMf8m
RaZWke69wKONkOxXcMO0h38lEBW3mg6RvZRbYPZvSLA9nxH7qBD7hTMA117mq2zoGh+WzY/Das5rq6mz5Eq7aeVKN8k2qS+DnAUC
ZVGDoi9vK+02opG5jR1gizqweeUqcRuzNyxCYfecNRWpVNYt4aXO8DwKmjJvt05F964DHkS1oUdFaOAehEOUe5B6a7YCuZq46MLu
2KgWIfgpIgSJI5mk1sGhR5aerqp2XiH/nSnvVDqq5paWbHA57gnfRJ8SQDXqlMGP+GSVaL1AOxd/61B20dMZiH4o3KmgUno7dYkE
peq2DR0dvHurbA2y2Tm8GQhXhuxfGhcaY0093lnvYhpTejVrio7s4Gj3zEu1tKN6NLMYFIYHPbU6QuUqpgNMx9saBrK9RjyKXI2K
MKXCBD6epZ44jE6d0waDcKuWSt5fkHPvCsrpwbtofEB4tF81M/A+LsR0oauzM2Qw6M3Q2A9snsvq9gUp5sZ8Sxr6plrDJgZeqK8p
fyDc0XytXLJajKLinaVLtiTgeWlHqv4Dd6j1b7M2G4dSdFjZpJrIPtLc53CqjgEsNXaGrz8eGL3pwoCuBszaqGeKFXESm1+1OJL9
zvfuPt/p48AORlHjg1OPJhZbu9sYn7cHMOgbXixoOVBqY7/K1aAsULsy+hBQalF038jfZynbiprkVu+7PtRYX4G/39pccm7bS7WD
AEpxsDZyvIf6b804O07AQ0BKQWh39sMx03iLcRQVD0HRwngNRCGfV6KwDuQXLPZSREsQ1jpXDWo6iAkb8NQnPEpMi/2Qqr04KB0z
UoV9aJHEalSdlzU2lekiBxe3GtbEsOjBbe2olUxTMW+bCCj32koIv9MSwu3+fc733Uk9qQ5RQzQzrBz591CRKCUHqJH3VJXzrfmr
D0GraVAMn66PbFfBgrrUqDMbw2I6dvTdo5klBb/ub0Rm0WxWUhC29q+lmSLyFimwPEhAn7ZOD/GKvEMjK3MO5a61xoGbbA9FoAJD
O8f9msEb+825543joZB9D83vuz85G14VrSfklYb4TWVwYO55p9fuMBA04NUK0XgVNQ70iusQjb/uYKe/Os/nE4ehJZJ1h1QqfYpv
q3+6q4ha5k3p28OnnD39oQIeSUOlDGQnElaivfSfPEgXvGDB218FBElP4dRUQ/HTVHMXD7c8Xt/KO2+RSRjCfsxDRwL25o2tMbrS
Yvvdn5r6Owfc0IhQY5ZsRIzjhdT8rQmxoEP8TTyeh8gwK2oUx+XcpBHQeYXCofwli+FQ9OXhkI6X4dC1f7aBXpiT/wPIKqw5MHEA
AA==
"""
open("agent.py", "wb").write(gzip.decompress(base64.b64decode("".join(B.split()))))

KN = sorted(f for f in glob.glob("chevalier*.*") if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")))
if not KN:
    raise SystemExit("❌ Image du chevalier introuvable : nomme-la chevalier.png (ou chevalier1.png, chevalier2.png…)")
MEDIA = (".jpg", ".jpeg", ".png", ".webp", ".mp4", ".mov")
if not os.listdir("fonds"):   # fonds posés à la racine du dépôt
    for f in os.listdir("."):
        if f.lower().endswith(MEDIA) and not f.startswith(("chevalier", "musique", "ma_voix")):
            shutil.move(f, "fonds/" + f)
if FONDS == "fichiers" and not os.listdir("fonds"):
    raise SystemExit("❌ Aucun arrière-plan trouvé : ajoute des images/vidéos, ou mets FONDS = \"genere\"")
if not os.path.exists("textes.txt"):
    open("textes.txt", "w", encoding="utf-8").write(TEXTES_PAR_DEFAUT)
MU = sorted(f for f in glob.glob("musique*.*") if f.lower().endswith((".mp3", ".m4a", ".wav", ".aac")))
print("   chevaliers :", KN, "| musiques :", MU, "| fonds :", len(os.listdir("fonds")))

print("3/5 Génération des vidéos (voix, sous-titres, montage)…")
KNS = ",".join(KN)
cmd = f"python agent.py --scripts textes.txt --bg fonds --knight {KNS} --knight_pos {CHEVALIER} --speed {VITESSE} --canvas {FORMAT} --accent {ACCENT} --bg_mode {FONDS} --bg_n {NB_IMAGES} --gen_engine {MOTEUR_IMAGES} --out sortie"
ref = next(iter(glob.glob("ma_voix.*")), None)
if CLONER and ref:
    cmd += f" --engine chatterbox --ref_voice {ref} --expressivity {EXPRESSIVITE} --deep 0"
else:
    cmd += f" --engine edge --voice {VOIX} --deep {GRAVE}"
if MU:
    cmd += " --music " + ",".join(MU)
sh(cmd)

print("4/5 Téléchargement…")
for f in sorted(glob.glob("sortie/*.mp4")):
    files.download(f)
print("5/5 ✅ Terminé")
